# Template


### Modules importation

In [ ]:
import timm
import torch
import torch.nn as nn
from timm.data.config import resolve_data_config # allow to correct inputs data by mean/std from the imported model
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

### Architecture

In [ ]:

class ZoobotConvNeXt(nn.Module):
    """Pretrained Encoder. Could be ConveNext or ResNEt """


    # 1. set up functions
    def __init__(
        self,
        num_classes: int = 3,
        model_name: str = "hf_hub:mwalmsley/zoobot-encoder-convnext_base", # this one lead to the best results on the zoobot benchmark
        pretrained: bool = True,
        freeze_backbone: bool = True,
        device: str | None = None,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu") # if you have a GPU it runs on GPU

        self.model = timm.create_model(
            model_name, pretrained=pretrained, num_classes=num_classes
        )
        # get the mean and std (parameters from pretrained model)
        self.data_config = resolve_data_config({}, model=self.model)

        if freeze_backbone: 
            self.freeze_backbone()
        self.to(self.device)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def freeze_backbone(self):
        for name, param in self.model.named_parameters():
            param.requires_grad = "head" in name

    def unfreeze_backbone(self):
        for param in self.model.parameters():
            param.requires_grad = True


    # 2. Data related functions
    def get_transforms(self, train: bool = True, size: int = 224): # Allow data augmentation
        mean, std = self.data_config["mean"], self.data_config["std"]
        if train:
            return transforms.Compose([
                transforms.Resize((size, size)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(180),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])
        return transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

    def get_dataloaders(self, root="data", batch_size=32, num_workers=0):
        """currently the standard folder tree expected is : root/train/<classe>/*.jpg et root/val/<classe>/*.jpg"""
        train_ds = datasets.ImageFolder(f"{root}/train", self.get_transforms(True))
        val_ds = datasets.ImageFolder(f"{root}/val", self.get_transforms(False))
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers
        )
        val_loader = DataLoader(
            val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers
        )
        return train_loader, val_loader

    # 3. Training functions 
    def fit(self, train_loader, val_loader=None, epochs=10, lr=1e-3, weight_decay=1e-2):
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(
            (p for p in self.parameters() if p.requires_grad),
            lr=lr,
            weight_decay=weight_decay,
        )
        history = {"train_loss": [], "val_loss": [], "val_acc": []}

        for epoch in range(epochs):
            self.train()
            running_loss = 0.0
            for images, labels in train_loader:
                images, labels = images.to(self.device), labels.to(self.device)

                optimizer.zero_grad()
                loss = criterion(self(images), labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

            train_loss = running_loss / len(train_loader)
            history["train_loss"].append(train_loss)
            msg = f"Epoch {epoch + 1}/{epochs} - train loss: {train_loss:.4f}"

            if val_loader is not None:
                val_loss, val_acc = self.evaluate(val_loader)
                history["val_loss"].append(val_loss)
                history["val_acc"].append(val_acc)
                msg += f" - val loss: {val_loss:.4f} - val acc: {val_acc:.4f}"
            print(msg)

        return history

    @torch.no_grad()
    def evaluate(self, loader):
        self.eval()
        criterion = nn.CrossEntropyLoss(reduction="sum")
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in loader:
            images, labels = images.to(self.device), labels.to(self.device)
            outputs = self(images)
            total_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
        return total_loss / total, correct / total

    # 4. Inference function 
    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> torch.Tensor:
        """Retourne les probabilités par classe pour un batch d'images."""
        self.eval()
        return torch.softmax(self(images.to(self.device)), dim=1)

    # 5. Load and save functions
    def save(self, path="zoobot_convnext.pth"):
        torch.save(self.state_dict(), path)

    def load(self, path="zoobot_convnext.pth"):
        self.load_state_dict(torch.load(path, map_location=self.device))

### Model set up

In [ ]:
# Parameters 
n_classes = 10
freeze_backbone = True
choosen_model = "https://huggingface.co/collections/mwalmsley/zoobot-encoders?sort=downloads"
pretrained = True

: 

In [ ]:
# Definition of the classifier
classifier =  ZoobotConvNeXt(num_classes=n_classes, freeze_backbone=freeze_backbone, model_name=choosen_model,pretrained=pretrained)

### Data importation

### Training

### Performance metrics